# CENG463 PA2

In this programming assignment, you will be dealing with word embeddings and neural networks. You will use Python for this task. You can use libraries such as `pandas`, `nltk`, `numpy` etc. for your implementations, or implement your own functions. However, you are expected to analyse and reason about your implementation and results. The assignment consists of 3 questions.

### IMPORTANT NOTE

Do not move or delete the given cells, only add cells inbetween the questions for your answers.

In [1]:
# UPDATE THIS CELL TO INSTALL NEEDED LIBRARIES.
# MAKE SURE TO ADD EVERYTHING THAT NEEDS TO BE INSTALLED IN THIS CELL!

# we will use pip to install packages - you can add others below
!pip install pandas
!pip install numpy
!pip install nltk
!pip install gensim
!pip install scikit-learn

# and import them here - you can add others below
import pandas as pd
import numpy as np
import nltk
import gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 6.7 MB/s  0:00:03m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [gensim]2m2/3 [gensim]


## Q1 - Word embeddings (50 points)

In this question, you will first train a Word2Vec model, then use it to represent and reason about user reviews.

### Q1.A - training (10 points)
Load the `user_review_train.csv` file shared with you. Using `Word2Vec` module of `gensim.models`, train a **skip-gram** Word2Vec model on the train data.

#### Notes and tips

- Use the given preprocessing function `preprocess_review`.

In [2]:
# PREPROCESSING FUNCTIONS GIVEN FOR YOU

from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('wordnet')  
nltk.download('stopwords')
nltk.download('punkt_tab')

def preprocess_review(review):
    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()
    sentences = sent_tokenize(review)
    
    lemmatized_review = []
    
    for sentence in sentences:
        tokenized_sentence = word_tokenize(sentence)
        lowercased_sentence = [token.lower() for token in tokenized_sentence]
        stopwords_removed_sentence = [token for token in lowercased_sentence if token not in stop_words]
        lemmatized_sentence = [lemmatizer.lemmatize(token) for token in stopwords_removed_sentence]
        
        lemmatized_review = lemmatized_review + lemmatized_sentence
    
    return lemmatized_review

[nltk_data] Downloading package wordnet to /Users/ovak/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to /Users/ovak/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /Users/ovak/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [10]:
# Q1.A - implementation
# you can add cells below if needed
review_test = pd.read_csv('data/user_review_train.csv')
review_test["preprocessed_review"] = review_test["review"].apply(preprocess_review)
print(review_test["preprocessed_review"].head())

model = gensim.models.Word2Vec(sentences=review_test["preprocessed_review"], 
                               vector_size=100, 
                               window=5, 
                               min_count=5,
                               workers=4, 
                               sg=1,
                               epochs=10)
print("Word2Vec model trained successfully.")
print(f"Vocabulary size: {len(model.wv)}")

0    [worst, mobile, bought, ever, ,, battery, drai...
1    [worst, phone, everthey, changed, last, phone,...
2    ['m, telling, n't, buyi, 'm, totally, disappoi...
3                               [battery, level, worn]
4    ['s, hitting, problem, ..., phone, hanging, pr...
Name: preprocessed_review, dtype: object


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Word2Vec model trained successfully.
Vocabulary size: 2539


### Q1.B - word similarity (10 points)

Using the trained model, report the following:

- Similarity between "good" and "bad"
- Similar words to "good"
- Similar words to "bad"
- Similar words to "good" but not similar to "bad"
- Similar words to "good" but not similar to "bad"

and discuss the reported words and scores. Is it possible to identify specific good/bad features of the product that is being reviewed? What other words can be looked up to get more insight?

#### Notes and tips

- Check the [documentation](https://tedboy.github.io/nlps/generated/generated/gensim.models.Word2Vec.html) of `gensim.models.Word2Vec` to find relevant methods.

In [11]:
# Q1.B - implementation
# you can add cells below if needed

sim = model.wv.similarity('good', 'bad')  # Example similarity check
print("Similarity between 'good' and 'bad':", sim)

most_similar_words = model.wv.most_similar("good")
print("Words most similar to 'good':", most_similar_words)

most_similar_words = model.wv.most_similar("bad")
print("Words most similar to 'bad':", most_similar_words)


simGoodbutNotBad = model.wv.most_similar(positive=['good'], negative=['bad'], topn=5)
print("Words similar to 'good' but not 'bad':", simGoodbutNotBad)

simGoodbutNotBad = model.wv.most_similar(positive=['bad'], negative=['good'], topn=5)
print("Words similar to 'bad' but not 'good':", simGoodbutNotBad)

Similarity between 'good' and 'bad': 0.54257
Words most similar to 'good': [('nice', 0.8342195749282837), ('satisfying', 0.8153236508369446), ('impressive', 0.7933875322341919), ('good.but', 0.7912298440933228), ('good.overall', 0.7888976335525513), ('beautiful', 0.7790176868438721), ('awesome', 0.7678866386413574), ('satisfactory', 0.7665425539016724), ('good.no', 0.7620073556900024), ('awsome', 0.7601497769355774)]
Words most similar to 'bad': [('pathetic', 0.6393648386001587), ('👎', 0.6331719160079956), ('bed', 0.6285042762756348), ('vary', 0.627659797668457), ('poor', 0.6250091791152954), ('worst', 0.5996263027191162), ('amozon', 0.5972906351089478), ('sad', 0.5797439217567444), ('disgusting', 0.5745508670806885), ('grt', 0.5693222284317017)]
Words similar to 'good' but not 'bad': [('operation', 0.4378490447998047), ('nice', 0.43528228998184204), ('liked', 0.4252406060695648), ('costly', 0.41453903913497925), ('drawback', 0.413878858089447)]
Words similar to 'bad' but not 'good': [

### Q1.B - discussion
Write your discussion here

### Q1.C - representation (15 points)

An important use of word embeddings is representing "documents" (reviews in our case). For this question, before creating the representations, do the following:

- Randomly sample 2 reviews from sentiment label 0, refer to them as sent0_a and sent0_b.
- Randomly sample 2 reviews from sentiment label 1, refer to them as sent1_a and sent1_b.

After the sampling, follow these steps to represent each review:

- Preprocess the review with the given `preprocess_review` function.
- For each token in the review, fetch the vector of that token.
- Take the average of the token vectors in the review to represent that review.

Then, calculate and report the cosine similarity of the two vectors representing:
    - sent0_a and sent0_b
    - sent0_a and sent1_a
    - sent1_a and sent1_b

Does this representation work to capture the labels of the reviews? Do you think there is a better way to represent each review instead of taking the average of the word vectors? Discuss your findings with respect to these questions. Repeating the sampling process several times might give you a better insight.

#### Notes and tips

- You can use `numpy` for your calculations.

In [22]:
# Q1.C - implementation
# you can add cells below if needed
def get_rndm_review(sentiment):
    while True:
        index = np.random.randint(0, len(review_test))
        select_review = review_test["review"][index]
        if(review_test["sentiment"][index] == sentiment):
            print("Selected review index:", index)
            break
    return select_review


np.random.seed(42)
sent1_a = get_rndm_review(1)
sent1_b = get_rndm_review(1)
sent0_a = get_rndm_review(0)
sent0_b = get_rndm_review(0)

print("Positive Review 1:", sent1_a)
print("Positive Review 2:", sent1_b)      
print("Negative Review 1:", sent0_a)
print("Negative Review 2:", sent0_b)

Selected review index: 13418
Selected review index: 11964
Selected review index: 5734
Selected review index: 6265
Positive Review 1: A good quality phone. Everything as expected. Camera is nice and battery lasts the entire day if not used for Hotspot.
Positive Review 2: Nice Product
Negative Review 1: Waist of money every time heating problem no battery back and in a one year 3 time repair on customer service center so please don't buy this
Negative Review 2: Worst of money to buy this product


In [23]:
#preprocess reviews

preprocessed_positive_review_1 = preprocess_review(sent1_a)
preprocessed_positive_review_2 = preprocess_review(sent1_b)
preprocessed_negative_review_1 = preprocess_review(sent0_a)
preprocessed_negative_review_2 = preprocess_review(sent0_b)

print("Preprocessed Positive Review 1:", preprocessed_positive_review_1)
print("Preprocessed Positive Review 2:", preprocessed_positive_review_2)
print("Preprocessed Negative Review 1:", preprocessed_negative_review_1)
print("Preprocessed Negative Review 2:", preprocessed_negative_review_2)


Preprocessed Positive Review 1: ['good', 'quality', 'phone', '.', 'everything', 'expected', '.', 'camera', 'nice', 'battery', 'last', 'entire', 'day', 'used', 'hotspot', '.']
Preprocessed Positive Review 2: ['nice', 'product']
Preprocessed Negative Review 1: ['waist', 'money', 'every', 'time', 'heating', 'problem', 'battery', 'back', 'one', 'year', '3', 'time', 'repair', 'customer', 'service', 'center', 'please', "n't", 'buy']
Preprocessed Negative Review 2: ['worst', 'money', 'buy', 'product']


In [26]:
#For each token in the review, fetch the vector of that token.

def get_review_vector(review):
    review_vector = np.zeros((100,))
    valid_word_count = 0
    for token in review:
        if token in model.wv:  # Check if token exists
            review_vector += model.wv[token]
            valid_word_count += 1
    
    if valid_word_count > 0:  # Avoid division by zero
        review_vector /= valid_word_count
    return review_vector

positive_review_vector_1 = get_review_vector(preprocessed_positive_review_1)
positive_review_vector_2 = get_review_vector(preprocessed_positive_review_2)
negative_review_vector_1 = get_review_vector(preprocessed_negative_review_1)
negative_review_vector_2 = get_review_vector(preprocessed_negative_review_2)

print("Positive Review Vector 1:", positive_review_vector_1)
print("Positive Review Vector 2:", positive_review_vector_2)
print("Negative Review Vector 1:", negative_review_vector_1)
print("Negative Review Vector 2:", negative_review_vector_2)

Positive Review Vector 1: [-0.16886367  0.31130691 -0.12405333  0.03532525  0.11628222 -0.14561666
  0.13910931  0.38602299 -0.21631806 -0.30561933 -0.05813731 -0.17365986
 -0.05214501 -0.06876592  0.16425142 -0.25780339  0.09252515 -0.22483219
 -0.1570355  -0.56553094  0.15192756  0.07696376  0.06558649 -0.2134606
 -0.20974113  0.00177546 -0.21270914 -0.32772055  0.06751615  0.09857845
  0.17848388  0.01764455  0.05045083 -0.22943281  0.00926711  0.25829649
 -0.05595805 -0.06505698  0.03715019 -0.20912168 -0.05540596 -0.128763
 -0.14186162  0.12454002  0.08913067 -0.10035543 -0.06583576 -0.1386634
 -0.00741133  0.17773046  0.12710497 -0.04309319 -0.02370258  0.04137825
 -0.00815924 -0.09776862  0.10538425 -0.05137862 -0.23530222 -0.01474707
  0.13659264 -0.01236524  0.12633708  0.06506854 -0.06295325  0.26705142
  0.21576959  0.15667641 -0.22629283  0.04130553  0.02233588  0.03064281
  0.03424949  0.04985797  0.1809014   0.14335046 -0.07876686  0.14916218
 -0.08476224 -0.10060404  0.0

In [27]:
# Cosine similarty function

def cosine_similarity(vec1, vec2):
    cos_sim = np.dot(vec1, vec2)/(np.linalg.norm(vec1)*np.linalg.norm(vec2))
    return cos_sim

# Calculate cosine similarities
sim_pos_pos = cosine_similarity(positive_review_vector_1, positive_review_vector_2)
sim_pos_neg_1 = cosine_similarity(positive_review_vector_1, negative_review_vector_1)
sim_neg_neg = cosine_similarity(negative_review_vector_1, negative_review_vector_2)


print("Cosine Similarity between two positive reviews:", sim_pos_pos)
print("Cosine Similarity between positive and negative review:", sim_pos_neg_1)
print("Cosine Similarity between two negative reviews:", sim_neg_neg)


Cosine Similarity between two positive reviews: 0.7472285420574502
Cosine Similarity between positive and negative review: 0.7493303889885714
Cosine Similarity between two negative reviews: 0.7264999099680947


### Q1.C - discussion
Write your discussion here

### Q1.D - training and comparing classifiers (15 points)

For this task, you will use the `user_review_train.csv` and `user_review_test.csv` files to train a binary classification model with Word2Vec representations, and compare its performance with a binary classifier using Bag-of-Words representation.

As the Bag-of-Words classifier, you can either choose the best performing classifier you have implemented in Question 3 of Programming Assignment 1, or you can follow these steps:

- Preprocess the review with the given `preprocess_review` function.
- Order all unique tokens by frequency, take the most frequent 100.
- Use these 100 words as the corpus for Bag-of-Words representation.

For the Word2Vec model, represent the reviews by following these steps:

- Preprocess the review with the given `preprocess_review` function.
- For each token in the review that is also in the most frequent 100 tokens, fetch the vector of that token.
- Take the average of the token vectors selected to represent that review.

After training both classifiers on `user_review_train.csv`, test them with `user_review_test.csv` and report the performance of your models with four metrics: accuracy, precision, recall and F1-score. Compare the performance of both models and discuss in detail.

#### Notes and tips

- You can use `CountVectorizer` from `scikit-learn` or any other library available for Bag-of-Words representation.
- You should select a classification method from the following set of classifiers: `[Naive Bayes, Support Vector Machine, Logistic Regression, Random Forest]`. You can use `scikit-learn`, `nltk`, or any other library for the classifier implementations. 
- You should **not** use the test set `user_reviews_test.csv` during your training process. You should use `user_reviews_train.csv` only.
- You may add a validation step in your training process. To do this, you can further split the `user_reviews_train.csv` data and apply k-fold cross validation.

## Q2 - Neural Networks for Binary Classification (50 points)

For this task, you will use the `user_review_train.csv` and `user_review_test.csv` files to train two neural network models for the binary classification of user reviews and compare their performances. You are expected to train RNN (part A - 20 points) and TextCNN (part B - 20 points) models, and report the following: 

- Confusion matrix of both models
- Time it took to train both models
- Accuracy, precision, recall, and F1-score of both models
- Other metrics you think are important

Finally (part C), you should discuss the performance of the models according to your reported results. Try to analyse the models in terms of pros and cons of using each one.

#### Notes and tips

- For the embedding layers of the models, you are free to use word embedding methods or leave them randomly initialised. Similarly, you can use word-based or character-based embeddings. However, make sure to explain your decisions.
- You are expected to use `tensorflow` for your implementations, but you can use other libraries if you already have a working setup.

In [ ]:
# Q2.A - implementation of RNN
# you can add cells below if needed



In [ ]:
# Q2.B - implementation of TextCNN
# you can add cells below if needed



### Q2.C - discussion

Write your discussion here.